# Download và chuẩn hóa PathVQA trên Kaggle

Notebook tự tải `flaviagiammarino/path-vqa` từ Hugging Face, lưu ảnh JPEG và tạo annotation gốc lẫn annotation dùng trực tiếp cho SelTDA.

Trước khi chạy, hãy bật **Internet** trong Kaggle Notebook. Notebook không cần GPU, không tải checkpoint và không tải pseudo-QA. Kết quả mặc định nằm trong `/kaggle/working/pathvqa`.

In [ ]:
from pathlib import Path

DATASET_ID = "flaviagiammarino/path-vqa"
OUTPUT_ROOT = Path("/kaggle/working/pathvqa")
JPEG_QUALITY = 95
OVERWRITE_IMAGES = False
CREATE_ARCHIVE = False

In [ ]:
import subprocess
import sys

packages = ["datasets", "huggingface-hub", "Pillow", "tqdm"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Đã cài dependency tải PathVQA.")

In [ ]:
import json
import re
import shutil
from PIL import Image
from tqdm.auto import tqdm

SPLIT_MAP = {"train": "train", "validation": "val", "test": "test"}


def infer_answer_type(answer):
    normalized = str(answer).lower().strip().rstrip(".")
    if normalized in {"yes", "no"}:
        return "yes/no"
    if re.fullmatch(r"[\d,.]+", normalized):
        return "number"
    return "other"


def infer_question_type(question, answer_type):
    if answer_type == "yes/no":
        return "yes/no"
    lowered = str(question).lower().strip()
    if lowered.startswith("how many") or lowered.startswith("how much"):
        return "how many"
    for prefix in ("what", "where", "why", "how", "when", "whose", "which", "who"):
        if lowered.startswith(prefix):
            return prefix
    return "other"


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, ensure_ascii=False),
        encoding="utf-8",
    )


def build_annotation_records(row, split_name, image_stem, question_id):
    question = str(row["question"]).strip()
    answer = str(row["answer"]).strip()
    answer_type = infer_answer_type(answer)
    question_type = infer_question_type(question, answer_type)

    raw_qa = {
        "image": image_stem,
        "question": question,
        "answer": answer,
    }
    raw_vqa = {
        "answer_type": answer_type,
        "img_id": image_stem,
        "label": {answer: 1},
        "question_id": question_id,
        "question_type": question_type,
        "sent": question,
    }
    converted = {
        "image": f"{split_name}/{image_stem}.jpg",
        "question": question,
        "answer": [answer] if split_name == "train" else answer,
        "dataset": "pathvqa",
        "question_id": question_id,
    }
    if split_name != "train":
        converted["question_type"] = question_type
        converted["answer_type"] = answer_type
    return raw_qa, raw_vqa, converted

In [ ]:
from datasets import load_dataset

print(f"Đang tải {DATASET_ID} từ Hugging Face...")
dataset = load_dataset(DATASET_ID)
missing_splits = set(SPLIT_MAP) - set(dataset)
if missing_splits:
    raise KeyError(f"Dataset thiếu split: {sorted(missing_splits)}")
print(dataset)

In [ ]:
def materialize_pathvqa(dataset, output_root):
    images_root = output_root / "images"
    raw_dump = {}
    converted_by_split = {}
    question_id = 0

    for hf_split, split_name in SPLIT_MAP.items():
        split_dir = images_root / split_name
        split_dir.mkdir(parents=True, exist_ok=True)
        raw_qa_records = []
        raw_vqa_records = []
        converted_records = []

        rows = dataset[hf_split]
        for index, row in enumerate(tqdm(rows, total=len(rows), desc=split_name)):
            image_stem = f"{split_name}_{index:06d}"
            image_path = split_dir / f"{image_stem}.jpg"
            if OVERWRITE_IMAGES or not image_path.exists():
                image = row["image"]
                if image.mode != "RGB":
                    image = image.convert("RGB")
                image.save(image_path, format="JPEG", quality=JPEG_QUALITY)

            raw_qa, raw_vqa, converted = build_annotation_records(
                row,
                split_name=split_name,
                image_stem=image_stem,
                question_id=question_id,
            )
            raw_qa_records.append(raw_qa)
            raw_vqa_records.append(raw_vqa)
            converted_records.append(converted)
            question_id += 1

        raw_dump[f"{split_name}_qa"] = raw_qa_records
        raw_dump[f"{split_name}_vqa"] = raw_vqa_records
        converted_by_split[split_name] = converted_records

    write_json(output_root / "all_data.json", raw_dump)
    for split_name in ("train", "val", "test"):
        write_json(output_root / f"{split_name}.json", converted_by_split[split_name])

    answer_list = sorted({record["answer"] for record in converted_by_split["test"]})
    write_json(output_root / "answer_list.json", answer_list)

    seen_images = set()
    test_val_combined = []
    for record in converted_by_split["val"] + converted_by_split["test"]:
        if record["image"] not in seen_images:
            seen_images.add(record["image"])
            test_val_combined.append(record)
    write_json(output_root / "test_val_combined.json", test_val_combined)
    return raw_dump, converted_by_split


OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
raw_dump, converted_by_split = materialize_pathvqa(dataset, OUTPUT_ROOT)
print(f"Đã ghi dữ liệu vào {OUTPUT_ROOT}")

In [ ]:
def validate_output(dataset, output_root, raw_dump, converted_by_split):
    summary = {}
    all_question_ids = []

    for hf_split, split_name in SPLIT_MAP.items():
        expected = len(dataset[hf_split])
        raw_qa = raw_dump[f"{split_name}_qa"]
        raw_vqa = raw_dump[f"{split_name}_vqa"]
        converted = converted_by_split[split_name]

        assert len(raw_qa) == expected, f"{split_name}: raw QA count mismatch"
        assert len(raw_vqa) == expected, f"{split_name}: raw VQA count mismatch"
        assert len(converted) == expected, f"{split_name}: converted count mismatch"

        missing_images = [
            record["image"]
            for record in converted
            if not (output_root / "images" / record["image"]).is_file()
        ]
        assert not missing_images, f"{split_name}: missing images: {missing_images[:5]}"

        all_question_ids.extend(record["question_id"] for record in converted)
        summary[split_name] = {
            "records": len(converted),
            "images": len(list((output_root / "images" / split_name).glob("*.jpg"))),
        }

    assert len(all_question_ids) == len(set(all_question_ids)), "question_id is not unique"

    expected_files = {
        "all_data.json",
        "train.json",
        "val.json",
        "test.json",
        "answer_list.json",
        "test_val_combined.json",
    }
    missing_files = [name for name in expected_files if not (output_root / name).is_file()]
    assert not missing_files, f"Missing output files: {missing_files}"
    return summary


summary = validate_output(dataset, OUTPUT_ROOT, raw_dump, converted_by_split)
print("Kiểm tra hoàn tất:")
for split_name, stats in summary.items():
    print(f"  {split_name}: {stats['records']:,} records, {stats['images']:,} images")

In [ ]:
from IPython.display import display

sample = converted_by_split["train"][0]
sample_image_path = OUTPUT_ROOT / "images" / sample["image"]
display(Image.open(sample_image_path))
print("Question:", sample["question"])
print("Answer:", sample["answer"][0])

In [ ]:
if CREATE_ARCHIVE:
    archive_base = OUTPUT_ROOT.parent / OUTPUT_ROOT.name
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_ROOT)
    print("Đã tạo archive:", archive_path)
else:
    print("CREATE_ARCHIVE=False, bỏ qua nén dữ liệu.")